# Week 7 — Final Dimensionality-Reduced Dataset
## Global Life Expectancy Paradox

This notebook combines feature selection and PCA outputs into a final CSV. It preserves the original cleaned columns, adds the derived average life-expectancy target and female–male gap, marks the top selected features, appends PCA component scores, and records feature-selection ranks.

**Design choices:** only the 15 original numeric indicators are used as candidate predictors; derived target fields and previously engineered/scaled duplicates are excluded from PCA. The number of components is selected to retain at least 90% of predictor variance.

In [ ]:
from google.colab import files
import io
uploaded = files.upload()  # Upload the supplied feature-engineered CSV.
filename = next(iter(uploaded))
import pandas as pd, numpy as np
df_all = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Uploaded shape:", df_all.shape)
display(df_all.head())

## Step 1 — Prepare original columns and target

In [ ]:
# Use only the original cleaned columns. Do not use precomputed *_scaled or *_minmax
# columns: doing so would duplicate information and distort PCA.
base_cols = [
    "name","region","iso2","population","pop_density","life_expectancy_male",
    "life_expectancy_female","sex_ratio","gdp_per_capita","gdp_growth",
    "unemployment","infant_mortality","fertility","homicide_rate",
    "urban_population","internet_users","secondary_school_enrollment_female",
    "secondary_school_enrollment_male","co2_emissions","forested_area"
]
df = df_all[base_cols].copy()
df["avg_life_expectancy"] = (df["life_expectancy_male"] + df["life_expectancy_female"]) / 2
target = "avg_life_expectancy"
predictors = [
    "population","pop_density","sex_ratio","gdp_per_capita","gdp_growth","unemployment",
    "infant_mortality","fertility","homicide_rate","urban_population","internet_users",
    "secondary_school_enrollment_female","secondary_school_enrollment_male",
    "co2_emissions","forested_area"
]
X = df[predictors].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
y = df[target].astype(float)
print(f"Rows: {len(df)} | candidate predictor features: {len(predictors)}")

## Step 2 — Select the top eight predictors

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
scores, pvals = f_regression(X, y)
selector = SelectKBest(score_func=f_regression, k=8).fit(X, y)
results = pd.DataFrame({
    "feature": predictors,
    "F_score": scores,
    "p_value": pvals,
    "correlation_with_target": [X[c].corr(y) for c in predictors],
    "selected_top_8": selector.get_support()
}).sort_values("F_score", ascending=False)
selected_features = results.loc[results["selected_top_8"], "feature"].tolist()
display(results)

## Step 3 — Standardize predictors and perform PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca_all = PCA().fit(X_scaled)
cumulative = np.cumsum(pca_all.explained_variance_ratio_)
n_components = int(np.searchsorted(cumulative, .90) + 1)
pca = PCA(n_components=n_components)
pc_scores = pca.fit_transform(X_scaled)
print(f"Using {n_components} components; retained variance = {pca.explained_variance_ratio_.sum():.2%}")

## Step 4 — Assemble and export final CSV

In [ ]:
final_df = df.copy()
for feature in predictors:
    final_df[f"selected_{feature}"] = int(feature in selected_features)
ranked = results.reset_index(drop=True)
rank_map = {row.feature: i+1 for i, row in enumerate(ranked.itertuples())}
for feature in predictors:
    final_df[f"{feature}_selection_rank"] = rank_map[feature]
for i in range(n_components):
    final_df[f"PC{i+1}"] = pc_scores[:, i]
final_df.to_csv("global_life_expectancy_dimensionality_reduced.csv", index=False)
results.to_csv("feature_selection_results.csv", index=False)
print("Final dataset shape:", final_df.shape)
display(final_df.head())
from google.colab import files
files.download("global_life_expectancy_dimensionality_reduced.csv")
files.download("feature_selection_results.csv")

## Step 5 — Interpretation and limitations
The selected features show univariate associations only. PCA components summarize variation among predictors, not causal drivers. Use a train/test split and fit selection/scaling/PCA on the training data only when evaluating predictive models.